# Lab 6: Randomness, Distributions, and Sampling

Welcome to Lab 6! This lab applies the material from **L11 (Probabilities and Randomness)** and **L12 (Sampling)** to two problems:

1. **Resistor quality control.** Simulate a production line with the normal distribution, measure its center and spread, and use the rules of probability to predict how many products pass inspection.
2. **Flight delays.** Treat every United flight out of SFO in summer 2015 as a population. Draw samples from it in different ways and see how well samples estimate the population.

After each question, run the `check(...)` cell below it to test your answer. If a check fails, it prints a hint.

First, run the cell below. It loads the libraries and the checker.

In [ ]:
# Run this cell first, but please don't change it.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import urllib.request
import importlib

# NumPy 2 prints numbers like np.float64(0.5); this makes them print as 0.5
try:
    np.set_printoptions(legacy="1.25")
except (ValueError, TypeError):
    pass

%matplotlib inline
plt.style.use('fivethirtyeight')

# All of the files for this lab live in the class GitHub folder.
# To read one, use: pd.read_csv(data_url + 'file_name.csv')
data_url = "https://raw.githubusercontent.com/madinyalab/engr10200/main/lab6/"

# Download the answer checker (one file) and load it
urllib.request.urlretrieve(data_url + "lab6_checks.py", "lab6_checks.py")
import lab6_checks
importlib.reload(lab6_checks)
from lab6_checks import check, check_all
print("Ready! Use check('q1_1') etc. to test your answers.")

## 1. Resistor Quality Control

Your company makes **100 Ω resistors**. The machine isn't perfect: the resistance of each part it makes is **normally distributed** with a mean of 100 Ω and a standard deviation of 2 Ω.

A customer will only accept resistors **between 96 Ω and 104 Ω** (inclusive). That range is the **spec**. Your job is to predict how many parts will pass, both for single resistors and for a circuit built from three of them.

Instead of measuring thousands of real resistors, we will **simulate** the production line.

**Question 1.1.** Simulate one production run of **10,000 resistors** with `np.random.normal`. Call `np.random.seed(2026)` first so your numbers match everyone else's. Then compute the mean, median, and standard deviation of the run.

Before you run it, think about how close `mean_R` and `median_R` should be for a normal distribution.

In [ ]:
np.random.seed(2026)
resistors = ...

mean_R = ...
median_R = ...
sd_R = ...
print(mean_R, median_R, sd_R)

In [ ]:
check("q1_1")

**Question 1.2.** You'll test "is it in spec?" many times in this lab, so write a function for it.

`in_spec(values, low, high)` should take an array and return a **boolean array** that is `True` wherever `low <= value <= high`. Give `low` a default of 96 and `high` a default of 104. Remember: to combine two conditions on arrays, use `&` and put each condition in parentheses.

Then:
* `frac_in_spec`: the fraction of `resistors` that are in spec
* `rule_estimate`: the fraction the **68-95-99.7 rule** predicts. (How many standard deviations from the mean are 96 and 104?)

In [ ]:
def in_spec(values, low=96, high=104):
    return (values >= ...) & (...)

frac_in_spec = np.sum(...) / len(resistors)
rule_estimate = ...
print(frac_in_spec, rule_estimate)

In [ ]:
check("q1_2")

**Question 1.3.** Make `R_bins`, an array of bin edges from 90 Ω **up to and including** 110 Ω, every 0.5 Ω. Then run the cell to plot a histogram of `resistors`. The red dashed lines, drawn with `plt.axvline`, mark the spec limits.

Roughly what fraction of the histogram lies between the red lines? Does that match your `frac_in_spec`?

In [ ]:
R_bins = ...

plt.hist(resistors, bins=R_bins, density=True, edgecolor='k')
plt.axvline(96, color='r', linestyle='--')
plt.axvline(104, color='r', linestyle='--')
plt.xlabel("Resistance (Ω)")
plt.ylabel("Probability density")
plt.title("Production run of 10,000 resistors")
plt.show()

In [ ]:
check("q1_3")

### From parts to products

One of your products is a circuit with **three resistors in series**, so its total resistance should be 300 Ω. On the assembly line, a worker grabs three resistors at random from the bin of `resistors`.

**Question 1.4.** Simulate **5,000** circuits. Use `np.random.choice(resistors, 5000)` to pick each of `r1`, `r2`, and `r3`, the first, second, and third resistor in every circuit. (The bin is so large that sampling with replacement is fine here.)

* `total`: the total resistance of each circuit
* `all_in`: a boolean array, `True` for circuits where **all three** resistors are in spec (use your `in_spec` function)
* `p_all_in_sim`: the fraction of simulated circuits where all three are in spec
* `p_all_in_rule`: the same probability, computed with the **multiplication rule**. Each resistor is in spec independently with probability `frac_in_spec`.

In [ ]:
r1 = ...
r2 = ...
r3 = ...
total = ...

all_in = ...
p_all_in_sim = ...
p_all_in_rule = ...
print(p_all_in_sim, p_all_in_rule)

In [ ]:
check("q1_4")

**Question 1.5.** The circuit has its own spec: the total must be between **297 Ω and 303 Ω**. Let $A$ = "the circuit is OK" and $B$ = "all three resistors are in spec".

* `circuit_ok`: boolean array of circuits whose `total` is in the circuit spec (reuse `in_spec` with new limits)
* `p_circuit_ok`: $P(A)$, the fraction of circuits that are OK
* `p_ok_given_all_in`: $P(A \mid B) = \dfrac{P(A \text{ and } B)}{P(B)}$
* `qc_guarantees_circuit`: `True` or `False`. If every resistor passes inspection, is the circuit guaranteed to be OK?

In [ ]:
circuit_ok = ...
p_circuit_ok = ...
p_ok_given_all_in = ...
qc_guarantees_circuit = ...
print(p_circuit_ok, p_ok_given_all_in)

In [ ]:
check("q1_5")

## 2. Flight Delays: Samples vs. the Population

The file `united_summer2015.csv` holds **every** United Airlines flight that left SFO between June 1 and August 9, 2015. Each row is one flight, and `Delay` is the departure delay in minutes (negative means it left early). Because we have every flight, this table is the **population**.

* A **parameter** is a number that describes the population, like the mean delay of all 13,825 flights.
* A **statistic** is a number computed from a sample, which we use to estimate a parameter.

In practice we usually can't see the whole population. Here we can, so we can check how well different samples do.

**Question 2.1.** Read the file into `united`. Then compute these population parameters for the `Delay` column:
* `pop_mean`, `pop_median`, and `pop_sd` (use `np.std`, which matches the formula from lecture)
* `pct_late_60`: the **percent** of flights delayed more than 60 minutes

Compare `pop_mean` and `pop_median`. What does that tell you about the shape of the distribution?

In [ ]:
united = ...

pop_mean = ...
pop_median = ...
pop_sd = ...
pct_late_60 = ...
print(pop_mean, pop_median, pop_sd, pct_late_60)

In [ ]:
check("q2_1")

In [ ]:
# Run this cell to see the population distribution.
delay_bins = np.arange(-20, 201, 5)
united.hist('Delay', bins=delay_bins, density=True)
plt.xlabel("Delay (minutes)")
plt.show()

**Question 2.2.** Draw three samples of about 139 flights each, using the three methods from lecture:
* `deterministic`: every 100th row, starting at row 0 (use `.take` and `np.arange`)
* `systematic`: every 100th row, starting at a **random** row from 0 to 99 (use `np.random.choice` to pick the start)
* `srs`: a simple random sample of 139 flights (use `.sample`)

Then fill in the `comparison` DataFrame. Its `'mean_delay'` column should hold each sample's mean delay, then `pop_mean`. The first one is done for you.

Run the cell several times. Set `never_changes` to the name of the method (as a string) whose mean is the same every time.

In [ ]:
deterministic = united.take(np.arange(..., ..., ...))

start = np.random.choice(np.arange(...))
systematic = united.take(np.arange(start, ..., ...))

srs = united.sample(...)

comparison = pd.DataFrame({
    'method': ['deterministic', 'systematic', 'random', 'population'],
    'mean_delay': [deterministic['Delay'].mean(), ..., ..., ...]
})
never_changes = ...
comparison

In [ ]:
check("q2_2")

**Question 2.3: The Law of Large Numbers.** Write a function `sample_mean_delay(n)` that draws a simple random sample of `n` flights and returns the mean delay of the sample.

Then compute `lln_means`, an array with one sample mean for each size in `sizes`. Plot `lln_means` against `sizes` with `plt.plot`, and draw a horizontal line at `pop_mean` with `plt.axhline(pop_mean, color='r')`. Label your axes.

What happens to the sample mean as the sample gets bigger?

In [ ]:
def sample_mean_delay(n):
    ...

sizes = np.arange(10, 3001, 10)
lln_means = ...

...
plt.show()

In [ ]:
check("q2_3")

**Question 2.4: Sample means are (nearly) normal.** The delays themselves are very skewed: most flights are close to on time, and a few are hours late. What happens if we take **many** samples and look at their means?

* `means_100`: an array of **1,000** sample means, each from a random sample of **100** flights (use `sample_mean_delay`)
* Plot a histogram of `means_100` with 30 bins and `density=True`.
* `mean_of_means` and `sd_of_means`: the mean and standard deviation (`np.std`) of `means_100`
* `frac_within_2sd`: the fraction of `means_100` within 2 standard deviations of `mean_of_means`. Use your `in_spec` function from Part 1 with new `low` and `high` limits.

Compare the shape of this histogram with the population histogram above. How is this like the dice averages from lecture?

In [ ]:
means_100 = ...

...
plt.show()

mean_of_means = ...
sd_of_means = ...
frac_within_2sd = ...
print(mean_of_means, sd_of_means, frac_within_2sd)

In [ ]:
check("q2_4")

## Done!

Run the cell below to check all of your answers at once.

In [ ]:
check_all()